In [1]:
# ============================================================
# Class-wise Physical Signal Plotting for UAV Propeller Dataset
# Author: Ajit and Partha
# Input file: merged_physics.csv
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ------------------------------------------------------------
# User settings
# ------------------------------------------------------------
DATA_FILE = "merged_physics.csv"
OUTPUT_DIR = "UAV_Classwise_Physical_Signal_Plots"
ACTIVE_RPM_MIN = 500
EPS = 1e-8

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Global plot style
# ------------------------------------------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["legend.fontsize"] = 12
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
df = pd.read_csv(DATA_FILE)
df.columns = df.columns.str.strip()

# ------------------------------------------------------------
# Helper function to identify column names safely
# ------------------------------------------------------------
def find_col(possible_names):
    """
    Finds the first matching column from a list of possible names.
    Matching is case-insensitive and ignores extra spaces.
    """
    col_map = {c.lower().strip(): c for c in df.columns}

    for name in possible_names:
        key = name.lower().strip()
        if key in col_map:
            return col_map[key]

    # partial matching
    for c in df.columns:
        c_low = c.lower()
        for name in possible_names:
            if name.lower() in c_low:
                return c

    raise ValueError(f"None of these columns found: {possible_names}")

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------
label_col = find_col(["class_label", "label", "class"])
rpm_col = find_col(["Motor Electrical Speed (RPM)", "RPM", "Motor Speed"])
thrust_col = find_col(["Thrust (gf)", "Thrust"])
torque_col = find_col(["Torque (N·m)", "Torque (Nm)", "Torque"])
current_col = find_col(["Current (A)", "Current"])
vibration_col = find_col(["Vibration (g)", "Vibration"])
power_col = find_col(["Electrical Power (W)", "Electrical Power", "Power"])

# ------------------------------------------------------------
# Clean labels and numerical values
# ------------------------------------------------------------
df[label_col] = df[label_col].astype(str).str.strip()

needed_cols = [
    label_col, rpm_col, thrust_col, torque_col,
    current_col, vibration_col, power_col
]

for col in needed_cols:
    if col != label_col:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=needed_cols)

# ------------------------------------------------------------
# Remove idle motor region
# ------------------------------------------------------------
df = df[df[rpm_col] > ACTIVE_RPM_MIN].copy()

# ------------------------------------------------------------
# Create thrust-per-watt feature
# ------------------------------------------------------------
df["Thrust_per_Watt_gf_per_W"] = df[thrust_col] / (df[power_col].abs() + EPS)

# Remove extreme ratio values only for clean visualization
lower_q = df["Thrust_per_Watt_gf_per_W"].quantile(0.01)
upper_q = df["Thrust_per_Watt_gf_per_W"].quantile(0.99)
df_plot = df[
    (df["Thrust_per_Watt_gf_per_W"] >= lower_q) &
    (df["Thrust_per_Watt_gf_per_W"] <= upper_q)
].copy()

# ------------------------------------------------------------
# Class order
# ------------------------------------------------------------
class_order = sorted(df_plot[label_col].unique())

# ------------------------------------------------------------
# Function: scatter plot
# ------------------------------------------------------------
def scatter_plot(data, x_col, y_col, xlabel, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(7.2, 5.4))

    for cls in class_order:
        temp = data[data[label_col] == cls]
        ax.scatter(
            temp[x_col],
            temp[y_col],
            s=28,
            alpha=0.70,
            edgecolors="none",
            label=str(cls)
        )

    ax.set_xlabel(xlabel, fontweight="bold")
    ax.set_ylabel(ylabel, fontweight="bold")
    ax.set_title(title, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(title="Class", frameon=True)
    plt.tight_layout()

    save_path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    return fig

# ------------------------------------------------------------
# Function: distribution plot
# ------------------------------------------------------------
def distribution_plot(data, feature_col, xlabel, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(7.2, 5.4))

    for cls in class_order:
        temp = data[data[label_col] == cls]
        ax.hist(
            temp[feature_col],
            bins=30,
            alpha=0.45,
            density=True,
            label=str(cls)
        )

    ax.set_xlabel(xlabel, fontweight="bold")
    ax.set_ylabel(ylabel, fontweight="bold")
    ax.set_title(title, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(title="Class", frameon=True)
    plt.tight_layout()

    save_path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    return fig

# ------------------------------------------------------------
# Generate figures
# ------------------------------------------------------------
figures = []

figures.append(
    scatter_plot(
        df_plot,
        rpm_col,
        thrust_col,
        "Motor Electrical Speed (RPM)",
        "Thrust (gf)",
        "Class-wise RPM vs Thrust",
        "Fig_RPM_vs_Thrust.pdf"
    )
)

figures.append(
    scatter_plot(
        df_plot,
        rpm_col,
        torque_col,
        "Motor Electrical Speed (RPM)",
        "Torque (N·m)",
        "Class-wise RPM vs Torque",
        "Fig_RPM_vs_Torque.pdf"
    )
)

figures.append(
    scatter_plot(
        df_plot,
        vibration_col,
        rpm_col,
        "Vibration (g)",
        "Motor Electrical Speed (RPM)",
        "Class-wise Vibration vs RPM",
        "Fig_Vibration_vs_RPM.pdf"
    )
)

figures.append(
    scatter_plot(
        df_plot,
        current_col,
        thrust_col,
        "Current (A)",
        "Thrust (gf)",
        "Class-wise Current vs Thrust",
        "Fig_Current_vs_Thrust.pdf"
    )
)

figures.append(
    distribution_plot(
        df_plot,
        "Thrust_per_Watt_gf_per_W",
        "Thrust per Electrical Watt (gf/W)",
        "Density",
        "Class-wise Thrust-per-Watt Distribution",
        "Fig_Thrust_per_Watt_Distribution.pdf"
    )
)

# ------------------------------------------------------------
# Save all plots into one combined multipage PDF
# ------------------------------------------------------------
combined_pdf_path = os.path.join(
    OUTPUT_DIR,
    "All_Classwise_Physical_Signal_Plots.pdf"
)

with PdfPages(combined_pdf_path) as pdf:
    for fig in figures:
        pdf.savefig(fig, bbox_inches="tight")

# Close all figures
for fig in figures:
    plt.close(fig)

# ------------------------------------------------------------
# Save summary text
# ------------------------------------------------------------
summary_path = os.path.join(OUTPUT_DIR, "plot_generation_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Class-wise physical signal plots generated successfully.\n\n")
    f.write(f"Input file: {DATA_FILE}\n")
    f.write(f"Output folder: {OUTPUT_DIR}\n")
    f.write(f"Active RPM threshold: RPM > {ACTIVE_RPM_MIN}\n\n")
    f.write("Generated PDF figures:\n")
    f.write("1. Fig_RPM_vs_Thrust.pdf\n")
    f.write("2. Fig_RPM_vs_Torque.pdf\n")
    f.write("3. Fig_Vibration_vs_RPM.pdf\n")
    f.write("4. Fig_Current_vs_Thrust.pdf\n")
    f.write("5. Fig_Thrust_per_Watt_Distribution.pdf\n")
    f.write("6. All_Classwise_Physical_Signal_Plots.pdf\n\n")
    f.write("Purpose:\n")
    f.write(
        "These plots support the physical interpretation of UAV propeller faults by "
        "showing how normal, bent, and cracked propellers differ in thrust, torque, "
        "RPM, vibration, current, and thrust-per-watt efficiency behaviour.\n"
    )

print("All plots saved successfully.")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Combined PDF: {combined_pdf_path}")

All plots saved successfully.
Output folder: UAV_Classwise_Physical_Signal_Plots
Combined PDF: UAV_Classwise_Physical_Signal_Plots\All_Classwise_Physical_Signal_Plots.pdf
